# NB06: Feature QC and sanitization

Sanitizes the patient-level OpenCLIP embeddings before training. Drops
all-NaN rows, drops columns with >2% missing, fills any residual NaNs
with column means, and drops near-constant columns.

**Manuscript reference** (Supplementary Table S8):
- Input feature dimensions: 512 (OpenCLIP ViT-B/16)
- Minimum non-NA column ratio: 0.98
- Minimum variance threshold: 1e-12
- Patients dropped (all-NaN): 1
- Columns dropped (missing): 0
- Columns dropped (constant): 0
- Imputed cells: 0
- Final feature dimensions: 512 (PCA reduces to 384 in NB07)

The `fillna(col_means)` step is a no-op for OpenCLIP ViT-B/16 because the
0.98 cutoff filters out any column with NaNs first; it is retained for
robustness when other backbones are substituted.

**Required env**: `WORKSPACE`.
**Inputs**: `embeddings/patient_means.parquet`, `labels/labels.parquet`.
**Outputs**: `embeddings/patient_means_clean.parquet`, sanitization report.

In [ ]:
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd

# env
WORKSPACE = Path(os.environ.get("WORKSPACE", "./workspace")).resolve()
EMB_DIR = WORKSPACE / "embeddings"
LABELS_DIR = WORKSPACE / "labels"
RESULTS_DIR = WORKSPACE / "results" / "qc"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# manuscript-locked sanitization knobs (Supp Table S8)
MIN_NON_NA_COL_RATIO = 0.98
MIN_VAR = 1e-12

print(f"WORKSPACE              : {WORKSPACE}")
print(f"min_non_na_col_ratio  : {MIN_NON_NA_COL_RATIO}")
print(f"min_var               : {MIN_VAR}")


In [ ]:
# load patient means and labels
emb_path = EMB_DIR / "patient_means.parquet"
labels_path = LABELS_DIR / "labels.parquet"
assert emb_path.exists(), f"missing {emb_path}; run NB03 first"
assert labels_path.exists(), f"missing {labels_path}; run NB04+NB05 first"

X = pd.read_parquet(emb_path).set_index("patient")
labels = pd.read_parquet(labels_path)
labels["patient"] = labels["patient"].astype(str).str.upper().str.slice(0, 12)
labels = labels.drop_duplicates("patient").set_index("patient")

print(f"raw embeddings : {X.shape}")
print(f"labels         : {labels.shape}")


In [ ]:
def sanitize_features(X, min_non_na_col_ratio=0.98, min_var=1e-12):
    pre = {
        "rows": int(X.shape[0]),
        "cols": int(X.shape[1]),
        "rows_all_nan": int(X.isna().all(axis=1).sum()),
        "cells_nan": int(X.isna().sum().sum()),
        "cells_inf": int(np.isinf(X.to_numpy(dtype=np.float64)).sum()),
    }

    # replace inf with NaN, then drop all-NaN rows
    X1 = X.replace([np.inf, -np.inf], np.nan)
    all_nan_rows = X1.isna().all(axis=1)
    X2 = X1.loc[~all_nan_rows].copy()
    dropped_row_ids = X.index[all_nan_rows].tolist()

    # drop columns with too many NaNs
    col_non_na = X2.notna().mean()
    keep_cols = col_non_na[col_non_na >= min_non_na_col_ratio].index
    drop_cols_missing = sorted(set(X2.columns) - set(keep_cols))
    X3 = X2[keep_cols].copy()

    # impute any residual NaNs with column means (no-op for OpenCLIP ViT-B/16)
    col_means = X3.mean(axis=0)
    na_before = int(X3.isna().sum().sum())
    X4 = X3.fillna(col_means)
    na_after = int(X4.isna().sum().sum())

    # drop near-constant columns
    var = X4.var(axis=0)
    keep_cols2 = var[var > min_var].index
    drop_cols_const = sorted(set(X4.columns) - set(keep_cols2))
    X5 = X4[keep_cols2].astype(np.float32)

    report = {
        "pre": pre,
        "dropped_rows_all_nan": int(all_nan_rows.sum()),
        "dropped_row_ids": dropped_row_ids,
        "dropped_cols_missing": len(drop_cols_missing),
        "dropped_cols_const": len(drop_cols_const),
        "imputed_cells": na_before - na_after,
        "final_shape": (int(X5.shape[0]), int(X5.shape[1])),
        "min_non_na_col_ratio": min_non_na_col_ratio,
        "min_var": min_var,
    }
    return X5, report


In [ ]:
X_clean, rep = sanitize_features(X, MIN_NON_NA_COL_RATIO, MIN_VAR)

print(f"raw shape    : {X.shape}")
print(f"clean shape  : {X_clean.shape}")
print(f"dropped rows : {rep['dropped_rows_all_nan']} (manuscript ref: 1)")
print(f"dropped cols : missing={rep['dropped_cols_missing']} const={rep['dropped_cols_const']} (manuscript refs: 0, 0)")
print(f"imputed cells: {rep['imputed_cells']} (manuscript ref: 0)")


In [ ]:
# write clean parquet and sanitization report
out_pq = EMB_DIR / "patient_means_clean.parquet"
X_clean.reset_index().to_parquet(out_pq, index=False)
(RESULTS_DIR / "sanitization_report.json").write_text(json.dumps(rep, indent=2))

# overlap with labels
common = X_clean.index.intersection(labels.index)
n_with_hrd = int(labels.loc[common, "HRD"].notna().sum()) if "HRD" in labels.columns else 0

summary = {
    "input_path": str(emb_path),
    "output_path": str(out_pq),
    "raw_shape": list(X.shape),
    "clean_shape": list(X_clean.shape),
    "patients_intersect_labels": int(len(common)),
    "patients_with_HRD": n_with_hrd,
    "manuscript_refs": {
        "input_dim": 512,
        "patients_dropped": 1,
        "cols_dropped_missing": 0,
        "cols_dropped_constant": 0,
        "imputed_cells": 0,
        "final_dim": 512,
        "labelled_patients": 8109,
    },
    "sanitization": rep,
}
(RESULTS_DIR / "summary.json").write_text(json.dumps(summary, indent=2, default=str))

print(json.dumps({k: v for k, v in summary.items() if k != "sanitization"}, indent=2))
print(f"\nwritten: {out_pq}")
